# HotpotQA · NARS + 8B + LoRA and Llama-3.1-8B (Colab, A100)

The only notebook that needs a GPU. One vLLM server holds Llama-3.1-8B-Instruct and the distilled adapter, so the
two systems share the same weights:
- **NARS + 8B + LoRA**: the adapter writes triples, aliases, plan and rules; ONA derives the answer.
- **Llama-3.1-8B**: the same 8B answers directly (the reference every other system is compared with).

**Before you run:** Runtime → **A100**; Colab secrets (🔑, *Notebook access* on): `DEEPINFRA_API_KEY` (the judge)
and `HF_TOKEN` (Hugging Face **read** token; the adapter is private). Then **Run all**. Setup about 10 minutes.

Answers go to Google Drive (`MyDrive/nars_lora/HotpotQA/results/`) as they finish; running again resumes. When it is
done, copy that folder's `answers/` and `traces/` into this repository's `HotpotQA/results/` (the baselines report then
includes these rows). The last cell releases the GPU.

In [ ]:
%%capture
!git clone -q https://github.com/MinaGabriel/NeuNARS.git /content/NeuNARS || git -C /content/NeuNARS pull -q
!git clone -q https://github.com/opennars/OpenNARS-for-Applications.git /content/ONA || true
!cd /content/ONA && git checkout -q dc4efd0abd520cdb79bf53bfa3c285ebb24f2e8a && sh build.sh
!pip install -q vllm prettytable

In [ ]:
import os, subprocess, sys, time, urllib.request
from google.colab import drive, userdata
from huggingface_hub import HfApi, snapshot_download

DATASET = "HotpotQA"
N_QUESTIONS = 200             # the first N questions; 5 for a quick check
WORKERS = 16

os.chdir(f"/content/NeuNARS/{DATASET}")
sys.path.insert(0, "/content/NeuNARS/common")
os.environ["DEEPINFRA_API_KEY"] = userdata.get("DEEPINFRA_API_KEY")
assert os.path.exists("/content/ONA/NAR"), "ONA did not build: run the first cell without %%capture to see why"
drive.mount("/content/drive")
RESULTS = f"/content/drive/MyDrive/nars_lora/{DATASET}/results"

import experiment as ex
revision = HfApi().model_info(ex.ADAPTER_REPO, token=userdata.get("HF_TOKEN")).sha
adapter = snapshot_download(ex.ADAPTER_REPO, revision=revision, token=userdata.get("HF_TOKEN"), local_dir="/content/adapter")
print(f"adapter {ex.ADAPTER_REPO} @ {revision[:8]}")

server = subprocess.Popen(
    ["vllm", "serve", "unsloth/Meta-Llama-3.1-8B-Instruct", "--served-model-name", "llama-3.1-8b-instruct",
     "--dtype", "bfloat16", "--enable-lora", "--lora-modules", f"nars-lora={adapter}", "--max-lora-rank", "16",
     "--max-model-len", "16384", "--gpu-memory-utilization", "0.90", "--port", "8000"],
    stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)
for minute in range(20):      # the first start downloads 16 GB of weights
    time.sleep(30)
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=5)
        print("vLLM is up")
        break
    except Exception:
        if server.poll() is not None:
            print(open("/content/vllm.log").read()[-3000:])
            raise SystemExit("vLLM stopped: see the log above")
        print(f"starting vLLM... {(minute + 1) / 2:.1f} min")

In [ ]:
from openai import OpenAI

K_VALUES = ex.K_VALUES[DATASET]
questions, question_hash = ex.load_questions("questions.pkl", N_QUESTIONS)
vllm = OpenAI(api_key="none", base_url="http://localhost:8000/v1", max_retries=2, timeout=600)
ex.check_models([(vllm, "nars-lora"), (vllm, "llama-3.1-8b-instruct"), (ex.deepinfra(), ex.JUDGE_MODEL)])
systems = {"Llama-3.1-8B": (vllm, "llama-3.1-8b-instruct", None),
           "NARS + 8B + LoRA": (vllm, "nars-lora", "/content/ONA/NAR")}
ex.run_all(questions, question_hash, RESULTS, systems, K_VALUES, workers=WORKERS)

## Report (these two systems)

In [ ]:
ex.report(RESULTS, DATASET, N_QUESTIONS)

## Done: release the GPU

In [ ]:
from google.colab import runtime
server.terminate()
drive.flush_and_unmount()
runtime.unassign()